# Шаг 5. Нарисуй цифру мышкой — сеть угадывает на лету

Проверим сеть на **своих** цифрах. Рисуем мышкой на чёрном холсте, а справа сеть показывает,
что она «видит» (картинку 28 × 28) и насколько уверена в каждой цифре.

Важный урок шага: сеть хорошо работает только на данных, **похожих на те, на которых училась**.
В MNIST цифры аккуратно подготовлены — вписаны в квадрат 20 × 20 и отцентрованы. Если подать сети
сырой рисунок, она часто ошибается. Галочка **«подготовка»** включает и выключает такую подготовку — сравните.

> **Это версия для Google Colab.** Холст появится прямо под ячейкой. На своём компьютере откройте `05_draw_digit.ipynb`.
> Нужна обученная сеть `mlp.pt` (ноутбук `03_train_mlp.ipynb`).

**Управление:** левая кнопка мыши — рисовать, кнопка **Стереть** — очистить холст, галочка — подготовка «как в MNIST» вкл/выкл.

**Структура ноутбука (алгоритм):**
1. Загружаем обученную сеть.
2. Рисование и кнопки работают в браузере (JavaScript), а считает сеть на сервере Colab (Python).
3. Функция `to_mnist` превращает холст в картинку 28 × 28 «как в MNIST»: обрезать → вписать в 20 × 20 → отцентровать.
4. Функция `predict` получает от сети 10 вероятностей.
5. Функция `panel` рисует правую панель: что видит сеть и столбики вероятностей.
6. Связываем браузер и Python: холст → подготовить → предсказать → показать панель.

In [ ]:
# Настройка для Google Colab. На своём компьютере эта ячейка ничего не делает.
# В Colab нужна папка machine_vision на Google Диске (в ней session01–session05 и dataset).
import os
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive:
    drive.mount("/content/drive")
    COLAB_ROOT = "/content/drive/MyDrive/machine_vision"
    os.chdir(f"{COLAB_ROOT}/session03")  # здесь лежит common.py и сохраняются результаты
    sys.path.insert(0, os.getcwd())

## 1. Импорт и сеть

- `SIZE` — размер холста: 420 × 420 пикселей. Рисуем белым — как в MNIST: белая цифра на чёрном фоне.
- `prepare` — флаг «включена подготовка как в MNIST».

In [ ]:
import base64

import cv2
import numpy as np
import torch
from google.colab import output
from IPython.display import HTML, JSON, display

from common import load_model

model = load_model()

SIZE = 420
prepare = True

## 2. Рисование мышью

Python в Colab работает на сервере и не видит вашу мышь, поэтому рисованием занимается **браузер**.
Холст и обработчики мыши написаны на JavaScript в разделе 6:
- кнопка нажата → начинаем линию;
- мышь движется с зажатой кнопкой → продолжаем линию;
- кнопка отпущена → перестаём.

Каждое изменение холста браузер отправляет в Python как картинку, а обратно получает готовую правую панель.

## 3. Подготовка «как в MNIST»

Создатели MNIST готовили каждую цифру так: вписывали её в квадрат 20 × 20 с сохранением пропорций и ставили
**центром масс** (средней точкой всех белых пикселей) в центр поля 28 × 28. Повторяем то же самое:

1. Если подготовка выключена — просто уменьшаем весь холст до 28 × 28 (`INTER_AREA` усредняет пиксели — подходит для уменьшения).
2. `np.nonzero(img)` — координаты всех белых пикселей. Если их нет — холст пустой, возвращаем чёрный квадрат.
3. **Обрезать** пустые поля: вырезаем прямоугольник от самого верхнего до самого нижнего и от самого левого до самого правого белого пикселя.
4. **Вписать в 20 × 20**: коэффициент `k = 20 / max(h, w)` — большая сторона станет 20, меньшая уменьшится пропорционально
   (`max(1, ...)` — защита от нулевого размера у очень тонкой линии).
5. Положить цифру в середину чёрного поля 28 × 28.
6. **Сдвинуть центр масс в точку (14, 14)**: `cv2.moments` считает «моменты» картинки; `m10 / m00` и `m01 / m00` —
   координаты центра масс по x и y. Матрица сдвига `[[1, 0, dx], [0, 1, dy]]` и `cv2.warpAffine` сдвигают картинку на `(dx, dy)`.

In [ ]:
def to_mnist(img):
    """Сделать из рисунка картинку 28×28, похожую на цифры MNIST.

    В MNIST цифра вписана в квадрат 20×20 и стоит центром масс посередине.
    Без этой подготовки сеть ошибается гораздо чаще — проверьте галочкой «подготовка».
    """
    if not prepare:
        return cv2.resize(img, (28, 28), interpolation=cv2.INTER_AREA)
    ys, xs = np.nonzero(img)
    if len(xs) == 0:
        return np.zeros((28, 28), np.uint8)
    digit = img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]  # обрезать пустые поля
    h, w = digit.shape
    k = 20 / max(h, w)
    digit = cv2.resize(digit, (max(1, round(w * k)), max(1, round(h * k))), interpolation=cv2.INTER_AREA)
    out = np.zeros((28, 28), np.uint8)
    h, w = digit.shape
    out[(28 - h) // 2:(28 - h) // 2 + h, (28 - w) // 2:(28 - w) // 2 + w] = digit
    m = cv2.moments(out)  # сдвинуть центр масс в точку (14, 14)
    shift = np.float32([[1, 0, 14 - m["m10"] / m["m00"]], [0, 1, 14 - m["m01"] / m["m00"]]])
    return cv2.warpAffine(out, shift, (28, 28))

## 4. Предсказание

Подаём картинку сети в том же виде, что при обучении:
1. `torch.tensor(..., dtype=torch.float32)` — массив NumPy в тензор с дробными числами;
2. `.div(255)` — диапазон 0…255 в 0…1 (так делал `ToTensor` при обучении);
3. `.view(1, 1, 28, 28)` — форма «пачка из 1 картинки, 1 канал, 28 × 28».

`torch.no_grad()` отключает подсчёт градиентов — при предсказании они не нужны, так быстрее.
`torch.softmax` превращает 10 оценок сети в **вероятности**: все положительные и в сумме дают 1.
`[0]` — первая (единственная) картинка пачки.

In [ ]:
def predict(small):
    x = torch.tensor(small, dtype=torch.float32).div(255).view(1, 1, 28, 28)
    with torch.no_grad():
        return torch.softmax(model(x), dim=1)[0].numpy()  # 10 вероятностей, в сумме 1

## 5. Правая панель

Функция `panel` рисует на тёмно-сером прямоугольнике 420 × 300:
1. **«Что видит сеть»** — картинку 28 × 28, увеличенную до 112 × 112. `INTER_NEAREST` копирует пиксели без сглаживания,
   поэтому видны отдельные «квадратики» — именно с таким разрешением работает сеть.
2. **10 столбиков** — по одному на цифру; длина столбика `probs[d] · 200` пикселей пропорциональна вероятности.
   Столбик самой вероятной цифры (`argmax`) — зелёный, остальные — голубые. Справа — вероятность в процентах.
3. Внизу — состояние подготовки `prepare: ON/OFF`.

Цвета в OpenCV задаются в порядке **(B, G, R)**: `(80, 200, 80)` — зелёный, `(0, 220, 255)` — жёлтый.

In [ ]:
def panel(small, probs):
    """Правая панель: что видит сеть (28×28) и столбики уверенности."""
    p = np.full((SIZE, 300, 3), 30, np.uint8)
    p[12:124, 94:206] = cv2.cvtColor(cv2.resize(small, (112, 112), interpolation=cv2.INTER_NEAREST),
                                     cv2.COLOR_GRAY2BGR)
    cv2.putText(p, "network sees", (96, 142), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)
    best = int(probs.argmax())
    for d in range(10):
        y = 168 + d * 24
        color = (80, 200, 80) if d == best else (200, 160, 90)
        cv2.putText(p, str(d), (14, y + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (230, 230, 230), 1)
        cv2.rectangle(p, (40, y + 2), (40 + int(probs[d] * 200), y + 18), color, -1)
        cv2.putText(p, f"{probs[d]:.0%}", (246, y + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (230, 230, 230), 1)
    cv2.putText(p, f"prepare: {'ON' if prepare else 'OFF'}", (14, SIZE - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 220, 255), 1)
    return p

## 6. Связываем браузер и Python

1. `on_draw` получает холст из браузера как картинку PNG, закодированную текстом (**base64**), и превращает её обратно в массив.
2. Дальше всё как раньше: `to_mnist` готовит холст, а `predict` спрашивает сеть, если на холсте что-то нарисовано (`canvas.any()`).
3. `panel` рисует правую панель, и она возвращается в браузер.
4. `output.register_callback` разрешает JavaScript вызывать функцию `on_draw` по имени `draw_digit.predict`.
5. `PAGE` — страница с холстом, панелью, кнопкой и галочкой. JavaScript разбирать не обязательно: главное происходит в `on_draw`.

In [ ]:
BRUSH = 16  # радиус кисти в пикселях


def on_draw(data_url, use_prepare):
    """Браузер прислал холст: готовим его, спрашиваем сеть, отправляем панель обратно."""
    global prepare
    prepare = use_prepare
    raw = base64.b64decode(data_url.split(",", 1)[1])
    canvas = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_GRAYSCALE)

    small = to_mnist(canvas)
    probs = predict(small) if canvas.any() else np.zeros(10)

    _, buf = cv2.imencode(".png", panel(small, probs))
    return JSON({"panel": "data:image/png;base64," + base64.b64encode(buf.tobytes()).decode()})


output.register_callback("draw_digit.predict", on_draw)

PAGE = """
<div style="display:flex;flex-wrap:wrap;gap:8px">
  <canvas id="board" width="SIZE" height="SIZE" style="cursor:crosshair;touch-action:none"></canvas>
  <img id="panel" width="300" height="SIZE">
</div>
<p>
  <button id="clear">Стереть</button>
  <label><input id="prepare" type="checkbox" checked> подготовка «как в MNIST»</label>
</p>
<script>
  const board = document.getElementById('board');
  const panel = document.getElementById('panel');
  const prepare = document.getElementById('prepare');
  const ctx = board.getContext('2d');
  let drawing = false, busy = false, again = false;

  function wipe() {
    ctx.fillStyle = 'black';
    ctx.fillRect(0, 0, SIZE, SIZE);
  }

  async function ask() {
    if (busy) { again = true; return; }
    busy = true;
    try {
      do {
        again = false;
        const answer = await google.colab.kernel.invokeFunction(
          'draw_digit.predict', [board.toDataURL('image/png'), prepare.checked], {});
        panel.src = answer.data['application/json'].panel;
      } while (again);
    } finally {
      busy = false;
    }
  }

  board.onpointerdown = (e) => {
    drawing = true;
    board.setPointerCapture(e.pointerId);
    ctx.beginPath();
    ctx.moveTo(e.offsetX, e.offsetY);
    board.onpointermove(e);
  };
  board.onpointermove = (e) => {
    if (!drawing) return;
    ctx.strokeStyle = 'white';
    ctx.lineWidth = 2 * BRUSH;
    ctx.lineCap = ctx.lineJoin = 'round';
    ctx.lineTo(e.offsetX, e.offsetY);
    ctx.stroke();
    ask();
  };
  board.onpointerup = () => { drawing = false; ask(); };
  document.getElementById('clear').onclick = () => { wipe(); ask(); };
  prepare.onchange = ask;

  wipe();
  ask();
</script>
""".replace("SIZE", str(SIZE)).replace("BRUSH", str(BRUSH))

display(HTML(PAGE))

## Попробуйте сами

1. Нарисуйте маленькую цифру в углу холста с выключенной подготовкой (снимите галочку), потом включите её. Что изменилось?
2. Нарисуйте «семёрку» с перечёркиванием, как пишут в России. Узнаёт ли её сеть? Почему?
3. Нарисуйте что-то, что вообще не цифра (букву, смайлик). Что ответит сеть? Может ли она сказать «не знаю»?
4. Поменяйте толщину кисти (`BRUSH = 16`) на 5 и 30. Как это влияет на ответы?